In [1]:
!pip install -q langchain langchain-community langchain-text-splitters
!pip install -q pypdf sentence-transformers faiss-cpu langchain-huggingface

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_871/2204708052.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
uploaded = files.upload()

pdf_path = next(iter(uploaded))

loader = PyPDFLoader(pdf_path)

documents = loader.load()

print(documents)

Saving BIODATA.pdf to BIODATA (1).pdf
[Document(metadata={'producer': 'Microsoft® Word 2024', 'creator': 'Microsoft® Word 2024', 'creationdate': '2026-11-11T21:39:58+05:30', 'author': 'Ananya mohan', 'moddate': '2026-11-11T21:39:58+05:30', 'source': 'BIODATA (1).pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content='BIODATA \n1. Personal Profile \nMy name is Ananya M. I am a Bachelor of Computer Applications (BCA) student specializing in \nArtificial Intelligence and Machine Learning (AIML) at Srinivas University, Bangalore Campus. I am an \nenthusiastic and hardworking student with an interest in technology, programming, artificial \nintelligence and data analytics. \nOver the past three years, I have developed my technical knowledge, communication skills, creativity \nand problem-solving abilities. I enjoy learning new technologies and applying my knowledge to \npractical projects. I believe in improving myself through continuous learning, teamwork and new \nexperiences

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 10


In [5]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

vector_store.save_local("ananya_biodata_index")

print("Data successfully indexed!")

Data successfully indexed!


In [7]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(
    "What are Ananya's academic projects?"
)

for result in results:
    print(result.page_content)

BIODATA 
1. Personal Profile 
My name is Ananya M. I am a Bachelor of Computer Applications (BCA) student specializing in 
Artificial Intelligence and Machine Learning (AIML) at Srinivas University, Bangalore Campus. I am an 
enthusiastic and hardworking student with an interest in technology, programming, artificial 
intelligence and data analytics. 
Over the past three years, I have developed my technical knowledge, communication skills, creativity
to data analytics concepts and learned how to apply theoretical knowledge to practical tasks. 
6. Achievements and Extracurricular Activities 
• Founding member of the Literary Club at Srinivas University. 
• Member of the Cultural Committee and Literary Committee. 
• Completed the Junior Examination in Bharatanatyam. 
• Participated in cultural activities and college events. 
• Developed teamwork, leadership, communication and stage presentation skills. 
7. Hobbies and Interests
3. Technical Skills 
• Programming: Python 
• Database: SQL 

In [8]:
!pip install -q langchain-groq

In [9]:
import os
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [10]:
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

Enter your Groq API Key: ··········


In [19]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

In [20]:
prompt = ChatPromptTemplate.from_template("""
You are Ananya's Personal AI Resume Assistant.

Answer the user's question using only the
provided biodata context.

If the answer is not available in the context,
say "This information is not available in my biodata."

Context:
{context}

Question:
{question}

Answer:
""")

In [21]:
def chatbot(question):

    # Retrieve relevant information
    docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Generate prompt
    messages = prompt.format_messages(
        context=context,
        question=question
    )

    # Generate answer
    response = llm.invoke(messages)

    return response.content

In [22]:
question = "What are Ananya's academic projects?"

answer = chatbot(question)

print("Question:", question)
print("Answer:", answer)

Question: What are Ananya's academic projects?
Answer: **Ananya's academic projects:**

1. **RIA – AI Interview Coach** – An AI-based interview preparation assistant that conducts mock interviews and evaluates performance.


In [23]:
!pip install -q gradio

In [24]:
import gradio as gr

def chat(message, history):
    return chatbot(message)

demo = gr.ChatInterface(
    fn=chat,
    title="Ananya's AI Resume Assistant",
    description="Ask me anything about Ananya's education, skills, projects and experience."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ea74938d2303359141.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
